# 🔧 Memory Test & Troubleshooting

Notebook này giúp kiểm tra và debug memory issues trước khi chạy evaluation notebooks.

## Chạy notebook này trước nếu:
- Kernel crashes khi chạy notebook 1 hoặc 2
- CUDA out of memory errors
- Muốn xem memory footprint của từng component

---

## 1. Check System Resources

In [ ]:
import torch
import psutil
import gc

print("="*80)
print("SYSTEM RESOURCES")
print("="*80)

# CPU
print(f"\n💻 CPU:")
print(f"   Cores: {psutil.cpu_count(logical=False)} physical, {psutil.cpu_count(logical=True)} logical")
print(f"   Usage: {psutil.cpu_percent()}%")

# RAM
ram = psutil.virtual_memory()
print(f"\n🧠 RAM:")
print(f"   Total: {ram.total / 1e9:.2f} GB")
print(f"   Available: {ram.available / 1e9:.2f} GB")
print(f"   Used: {ram.used / 1e9:.2f} GB ({ram.percent}%)")

# GPU
print(f"\n🎮 GPU:")
if torch.cuda.is_available():
    print(f"   Available: ✅ YES")
    print(f"   Device: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"   Total Memory: {props.total_memory / 1e9:.2f} GB")
    print(f"   Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"   Cached: {torch.cuda.memory_reserved() / 1e9:.2f} GB")
    print(f"   Free: {(props.total_memory - torch.cuda.memory_allocated()) / 1e9:.2f} GB")
else:
    print(f"   Available: ❌ NO (will use CPU)")

print("\n" + "="*80)

## 2. Memory Recommendations

In [ ]:
print("\n" + "="*80)
print("RECOMMENDATIONS")
print("="*80)

recommendations = []

# Check RAM
if ram.available / 1e9 < 4:
    recommendations.append("⚠️ Low RAM (<4GB). Close other applications.")
elif ram.available / 1e9 < 8:
    recommendations.append("⚠️ Moderate RAM (<8GB). Consider reducing sample size.")
else:
    recommendations.append("✅ RAM sufficient (>8GB available)")

# Check GPU
if torch.cuda.is_available():
    gpu_mem = props.total_memory / 1e9
    if gpu_mem < 6:
        recommendations.append("⚠️ Low GPU memory (<6GB). Reduce SAMPLE_QUERIES_PER_DATASET to 15.")
        recommendations.append("   Consider running on CPU instead.")
    elif gpu_mem < 8:
        recommendations.append("⚠️ Moderate GPU memory (<8GB). Reduce SAMPLE_QUERIES_PER_DATASET to 20.")
    else:
        recommendations.append("✅ GPU memory sufficient (>8GB)")
else:
    recommendations.append("ℹ️ No GPU. Evaluation will run on CPU (slower but stable).")

for rec in recommendations:
    print(f"\n{rec}")

print("\n" + "="*80)

## 3. Test Small Scale Evaluation

Run a mini version to check if everything works:

In [ ]:
import sys
sys.path.insert(0, '..')

from sentence_transformers import SentenceTransformer
from FlagEmbedding import FlagReranker
import faiss
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print("\nTesting component memory footprint...\n")

# Test 1: Embedding model
print("1. Loading embedding model...")
try:
    embed_model = SentenceTransformer('intfloat/e5-small-v2', device=device)
    
    # Test encoding
    texts = ["This is a test sentence."] * 100
    embeddings = embed_model.encode(texts, show_progress_bar=False, convert_to_numpy=True)
    
    if torch.cuda.is_available():
        mem_used = torch.cuda.memory_allocated() / 1e9
        print(f"   ✅ Success. GPU memory used: {mem_used:.2f} GB")
    else:
        print(f"   ✅ Success (CPU mode)")
    
    del embed_model, embeddings
    if device == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()
    
except Exception as e:
    print(f"   ❌ Failed: {e}")

# Test 2: Reranker model
print("\n2. Loading reranker model...")
try:
    reranker = FlagReranker('BAAI/bge-reranker-base', use_fp16=(device=='cuda'))
    
    # Test reranking
    pairs = [["query", "document"]] * 10
    scores = reranker.compute_score(pairs)
    
    if torch.cuda.is_available():
        mem_used = torch.cuda.memory_allocated() / 1e9
        print(f"   ✅ Success. GPU memory used: {mem_used:.2f} GB")
    else:
        print(f"   ✅ Success (CPU mode)")
    
    del reranker
    if device == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()
    
except Exception as e:
    print(f"   ❌ Failed: {e}")

# Test 3: FAISS index
print("\n3. Testing FAISS index...")
try:
    # Create dummy embeddings
    dim = 384
    n = 10000
    embeddings = np.random.randn(n, dim).astype('float32')
    
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    
    # Test search
    query_emb = np.random.randn(1, dim).astype('float32')
    scores, indices = index.search(query_emb, 100)
    
    print(f"   ✅ Success. Indexed {n} vectors.")
    
    del index, embeddings
    gc.collect()
    
except Exception as e:
    print(f"   ❌ Failed: {e}")

print("\n" + "="*80)
print("✅ All components tested successfully!")
print("   You can proceed with the main evaluation notebooks.")
print("="*80)

## 4. Config Recommendations

In [ ]:
print("\n" + "="*80)
print("RECOMMENDED CONFIG.PY SETTINGS")
print("="*80)

if torch.cuda.is_available():
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    
    if gpu_mem < 6:
        print("\n⚠️ Low GPU Memory (<6GB):")
        print("   SAMPLE_QUERIES_PER_DATASET = 15")
        print("   TOP_K_RETRIEVAL = 50")
        print("   TOP_K_RERANK = 30")
        print("   EMBED_BATCH_SIZE = 16")
        print("   RERANK_BATCH_SIZE = 8")
    elif gpu_mem < 8:
        print("\n⚠️ Moderate GPU Memory (6-8GB):")
        print("   SAMPLE_QUERIES_PER_DATASET = 20")
        print("   TOP_K_RETRIEVAL = 75")
        print("   TOP_K_RERANK = 40")
        print("   EMBED_BATCH_SIZE = 24")
        print("   RERANK_BATCH_SIZE = 12")
    else:
        print("\n✅ High GPU Memory (>8GB):")
        print("   SAMPLE_QUERIES_PER_DATASET = 30  (default)")
        print("   TOP_K_RETRIEVAL = 100  (default)")
        print("   TOP_K_RERANK = 50  (default)")
        print("   EMBED_BATCH_SIZE = 32  (default)")
        print("   RERANK_BATCH_SIZE = 16  (default)")
else:
    print("\nℹ️ CPU Mode (No GPU):")
    print("   SAMPLE_QUERIES_PER_DATASET = 20  (reduced for speed)")
    print("   TOP_K_RETRIEVAL = 50")
    print("   TOP_K_RERANK = 30")
    print("   EMBED_BATCH_SIZE = 16")
    print("   RERANK_BATCH_SIZE = 8")
    print("   ⚠️ Expect 3-5x slower than GPU")

print("\n" + "="*80)

## 5. Clear All Memory

In [ ]:
# Force clear all cached memory
import gc
import torch

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print("✅ GPU memory cleared")
    print(f"   Current allocation: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
else:
    print("✅ Memory cleared (CPU mode)")

print("\n🚀 Ready to run evaluation notebooks!")